In [38]:
import pickle
import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torch.utils.data import (
    Dataset,
    DataLoader
)

from sklearn.model_selection import (
    train_test_split
)

from tqdm import tqdm

In [39]:
positions_df = pd.read_parquet(
    "Data/positions.parquet"
)

print(
    positions_df.shape
)

positions_df.head()

(2967604, 6)


,game_id,ply,board_fen,move,result,token_ids
0,1,0,rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR,e2e4,0-1,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,..."
1,1,1,rnbqkbnr/pppppppp/8/8/4P3/8/PPPP1PPP/RNBQKBNR,d7d6,0-1,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,..."
2,1,2,rnbqkbnr/ppp1pppp/3p4/8/4P3/8/PPPP1PPP/RNBQKBNR,d2d4,0-1,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 13, 14,..."
3,1,3,rnbqkbnr/ppp1pppp/3p4/8/3PP3/8/PPP2PPP/RNBQKBNR,g8f6,0-1,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 13, 14,..."
4,1,4,rnbqkb1r/ppp1pppp/3p1n2/8/3PP3/8/PPP2PPP/RNBQKBNR,b1c3,0-1,"[0, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 13, 14, 15..."


In [40]:
with open(
    "vocab.pkl",
    "rb"
) as f:

    vocab = pickle.load(f)


with open(
    "move_vocab.pkl",
    "rb"
) as f:

    move_vocab = pickle.load(f)


with open(
    "idx_to_token.pkl",
    "rb"
) as f:

    idx_to_token = pickle.load(f)

In [41]:
v2_embeddings = torch.load(
    "piece_embeddings.pt",
    map_location="cpu"
)

print(
    v2_embeddings.shape
)

torch.Size([737, 128])


In [42]:
idx_to_move = {

    idx: move

    for move, idx

    in move_vocab.items()

}

In [43]:
MAX_PIECES = 32

PAD_ID = len(vocab)

vocab_size = PAD_ID + 1

num_moves = len(move_vocab)

print(vocab_size)
print(num_moves)

737
1896


In [44]:
print(
    type(
        positions_df["token_ids"].iloc[0]
    )
)

print(
    positions_df["token_ids"].iloc[0]
)

<class 'numpy.ndarray'>
[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29 30 31]


In [45]:
X = np.full(

    (
        len(positions_df),
        MAX_PIECES
    ),

    PAD_ID,

    dtype=np.int64
)

In [46]:
for i, tokens in enumerate(
    positions_df["token_ids"]
):

    n = min(
        len(tokens),
        MAX_PIECES
    )

    X[i,:n] = tokens[:n]

In [47]:
positions_df["move_id"] = (
    positions_df["move"]
    .map(move_vocab)
)

In [48]:
y = positions_df[
    "move_id"
].values

In [49]:
X = torch.tensor(
    X,
    dtype=torch.long
)

y = torch.tensor(
    y,
    dtype=torch.long
)

In [50]:
unique_games = (
    positions_df["game_id"]
    .unique()
)

In [51]:
train_games, val_games = (
    train_test_split(
        unique_games,
        test_size=0.1,
        random_state=42
    )
)

In [52]:
train_mask = (
    positions_df["game_id"]
    .isin(train_games)
)

val_mask = (
    positions_df["game_id"]
    .isin(val_games)
)

In [53]:
train_idx.dtype

dtype('int64')

In [54]:
val_idx.dtype

dtype('int64')

In [58]:
train_idx = torch.tensor(
    np.where(train_mask.values)[0],
    dtype=torch.long
)

val_idx = torch.tensor(
    np.where(val_mask.values)[0],
    dtype=torch.long
)

X_train = X[train_idx]
y_train = y[train_idx]

X_val = X[val_idx]
y_val = y[val_idx]

print(X_train.shape)
print(X_val.shape)

torch.Size([2669196, 32])
torch.Size([298408, 32])


In [59]:
from torch.utils.data import Dataset

class ChessDataset(Dataset):

    def __init__(self, X, y):

        self.X = X
        self.y = y

    def __len__(self):

        return len(self.X)

    def __getitem__(self, idx):

        return (
            self.X[idx],
            self.y[idx]
        )

In [60]:
train_dataset = ChessDataset(
    X_train,
    y_train
)

val_dataset = ChessDataset(
    X_val,
    y_val
)

print(len(train_dataset))
print(len(val_dataset))

2669196
298408


In [61]:
from torch.utils.data import DataLoader

BATCH_SIZE = 1024

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

In [62]:
batch_x, batch_y = next(
    iter(train_loader)
)

batch_x = batch_x.to(device)

logits, attn = model(batch_x)

print(logits.shape)
print(attn.shape)

torch.Size([1024, 1896])
torch.Size([1024, 33, 33])


**V3 Model Making**

Above was pre processing


In [63]:
import torch
import torch.nn as nn
class ChessAttentionModel(nn.Module):

    def __init__(
        self,
        vocab_size,
        num_moves,
        embedding_dim=128,
        num_heads=8
    ):

        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=PAD_ID
        )

        # CLS token
        self.cls_token = nn.Parameter(
            torch.randn(
                1,
                1,
                embedding_dim
            )
        )

        self.attention = nn.MultiheadAttention(
            embed_dim=embedding_dim,
            num_heads=num_heads,
            batch_first=True
        )

        self.norm = nn.LayerNorm(
            embedding_dim
        )

        self.fc = nn.Sequential(

            nn.Linear(
                embedding_dim,
                256
            ),

            nn.ReLU(),

            nn.Dropout(0.2),

            nn.Linear(
                256,
                num_moves
            )
        )

    def forward(
        self,
        x
    ):

        batch_size = x.shape[0]

        mask = (
            x == PAD_ID
        )

        x = self.embedding(x)

        cls = self.cls_token.expand(
            batch_size,
            -1,
            -1
        )

        x = torch.cat(
            [cls, x],
            dim=1
        )

        cls_mask = torch.zeros(
            batch_size,
            1,
            dtype=torch.bool,
            device=x.device
        )

        mask = torch.cat(
            [cls_mask, mask],
            dim=1
        )

        attn_out, attn_weights = self.attention(
            x,
            x,
            x,
            key_padding_mask=mask
        )

        attn_out = self.norm(
            attn_out
        )

        board_vector = attn_out[:, 0]

        logits = self.fc(
            board_vector
        )

        return logits, attn_weights

In [64]:
import torch

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Using:", device)

if torch.cuda.is_available():
    print(
        torch.cuda.get_device_name(0)
    )

Using: cpu


In [65]:
model = ChessAttentionModel(
    vocab_size=vocab_size,
    num_moves=num_moves
)

model.to(device)

ChessAttentionModel(
  (embedding): Embedding(737, 128, padding_idx=736)
  (attention): MultiheadAttention(
    (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
  )
  (norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
  (fc): Sequential(
    (0): Linear(in_features=128, out_features=256, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=256, out_features=1896, bias=True)
  )
)

ChessAttentionModel(
  (embedding): Embedding(737, 128, padding_idx=736)
  (attention): MultiheadAttention(
    (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
  )
  (norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
  (fc): Sequential(
    (0): Linear(in_features=128, out_features=256, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=256, out_features=1896, bias=True)
  )
)
Result Frm Above


In [66]:
with torch.no_grad():

    model.embedding.weight.copy_(
        v2_embeddings
    )

# Model.Embedding.Weight.Shape Is torch.Size([737, 128])

In [67]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4
)

In [68]:
batch_x, batch_y = next(
    iter(train_loader)
)

batch_x = batch_x.to(device)

logits, attn = model(
    batch_x
)

print(logits.shape)
print(attn.shape)

torch.Size([1024, 1896])
torch.Size([1024, 33, 33])


In [ ]:
from tqdm import tqdm

model.train()

running_loss = 0

for batch_x, batch_y in tqdm(
    train_loader
):

    batch_x = batch_x.to(device)

    batch_y = batch_y.to(device)

    optimizer.zero_grad()

    logits, _ = model(
        batch_x
    )

    loss = criterion(
        logits,
        batch_y
    )

    loss.backward()

    optimizer.step()

    running_loss += (
        loss.item()
    )

print(
    "Average Loss:",
    running_loss /
    len(train_loader)
)

  2%|▏         | 48/2607 [00:26<29:21,  1.45it/s]

In [ ]:
torch.save(
    model.state_dict(),
    "v3_attention_model.pt"
)

In [ ]:
model.eval()

batch_x, batch_y = next(
    iter(val_loader)
)

with torch.no_grad():

    logits, attn = model(
        batch_x
    )

print(
    attn.shape
)

In [ ]:
attn[0]